# Proyecto Grupo 27 - Visualizacion de Informacion
## Comparacion del consumo de energia en America: Pandemia vs Post-pandemia

Este notebook analiza datos de generacion electrica en paises de America, comparando dos periodos:
- **Pandemia**: 2020 - 2022
- **Post-pandemia**: 2023 - 2025

**Dataset:** `release_generation_yearly_global.csv` (fuente: Ember Global Electricity Review)

---

## Paso 1: Importar librerias
Importamos `pandas` para la manipulacion y analisis de datos tabulares.

In [ ]:
import pandas as pd

## Paso 2: Cargar el dataset
Leemos el CSV completo y mostramos informacion basica: cantidad de filas, columnas y las primeras filas del dataset para entender su estructura.

In [ ]:
ruta_csv = "release_generation_yearly_global.csv"
df = pd.read_csv(ruta_csv)

print(f"Filas totales: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print(f"\nColumnas disponibles:")
for col in df.columns:
    print(f"  - {col}")

df.head()

## Paso 3: Filtrar solo paises de America
El dataset contiene datos de todo el mundo. Filtramos unicamente los paises que pertenecen a **North America** y **South America** segun la columna `Continent`.

Ademas, excluimos las filas de tipo `Region` (como "ASEAN", "Latin America", etc.) y nos quedamos solo con `Country or economy`.

In [ ]:
df_americas = df[
    (df["Continent"].isin(["North America", "South America"])) &
    (df["Area type"] == "Country or economy")
].copy()

paises_america = sorted(df_americas["Area"].unique())
print(f"Paises de America encontrados: {len(paises_america)}\n")

for pais in paises_america:
    iso = df_americas[df_americas["Area"] == pais]["ISO 3 code"].iloc[0]
    print(f"  - {pais} ({iso})")

## Paso 4: Definir los periodos de analisis
Separamos los datos en dos rangos de anios:
- **Pandemia (2020-2022):** periodo donde el COVID-19 impacto el consumo energetico.
- **Post-pandemia (2023-2025):** periodo de recuperacion y normalizacion.

In [ ]:
PERIODO_PANDEMIA = [2020, 2021, 2022]
PERIODO_POST_PANDEMIA = [2023, 2024, 2025]

print(f"Periodo Pandemia:      {PERIODO_PANDEMIA}")
print(f"Periodo Post-pandemia: {PERIODO_POST_PANDEMIA}")

## Paso 5: Filtrar generacion total y separar por periodo
De todas las fuentes de electricidad (Solar, Wind, Coal, Gas, etc.), nos quedamos unicamente con la fila **"Total generation"**, que representa el consumo/generacion total de energia de cada pais por anio.

Luego separamos esos datos en los dos periodos definidos.

In [ ]:
# Filtrar solo la generacion total por pais/anio
df_total_gen = df_americas[
    df_americas["Electricity source"] == "Total generation"
].copy()

# Datos durante la pandemia (2020-2022)
df_pandemia = df_total_gen[
    df_total_gen["Year"].isin(PERIODO_PANDEMIA)
].copy()

# Datos post-pandemia (2023-2025)
df_post_pandemia = df_total_gen[
    df_total_gen["Year"].isin(PERIODO_POST_PANDEMIA)
].copy()

print(f"Registros America (todos los anios):  {len(df_total_gen):,}")
print(f"Registros Pandemia (2020-2022):       {len(df_pandemia):,}")
print(f"Registros Post-pandemia (2023-2025):  {len(df_post_pandemia):,}")

Veamos como se ven los datos de pandemia (primeras filas):

In [ ]:
df_pandemia[["Area", "ISO 3 code", "Year", "Generation (TWh)", "Continent"]].head(10)

Y los datos post-pandemia (primeras filas):

In [ ]:
df_post_pandemia[["Area", "ISO 3 code", "Year", "Generation (TWh)", "Continent"]].head(10)

## Paso 6: Calcular promedios por periodo
Para cada pais, calculamos el **promedio de generacion electrica (TWh)** durante la pandemia y durante el periodo post-pandemia. Esto nos da un valor representativo de cada periodo para poder compararlos.

In [ ]:
# Promedios pandemia por pais
promedio_pandemia = (
    df_pandemia.groupby(["Area", "ISO 3 code", "Continent"])["Generation (TWh)"]
    .mean()
    .reset_index()
    .rename(columns={"Generation (TWh)": "Gen_promedio_pandemia_TWh"})
)

# Promedios post-pandemia por pais
promedio_post_pandemia = (
    df_post_pandemia.groupby(["Area", "ISO 3 code", "Continent"])["Generation (TWh)"]
    .mean()
    .reset_index()
    .rename(columns={"Generation (TWh)": "Gen_promedio_post_pandemia_TWh"})
)

print("Promedios pandemia (muestra):")
promedio_pandemia.head()

## Paso 7: Crear tabla comparativa
Unimos ambos DataFrames de promedios en una sola tabla comparativa. Ademas calculamos:
- **Diferencia (TWh):** cuanto subio o bajo la generacion entre periodos.
- **Cambio porcentual (%):** la variacion porcentual entre ambos periodos.

In [ ]:
df_comparacion = promedio_pandemia.merge(
    promedio_post_pandemia,
    on=["Area", "ISO 3 code", "Continent"],
    how="outer"
)

# Calcular diferencia y cambio porcentual
df_comparacion["Diferencia_TWh"] = (
    df_comparacion["Gen_promedio_post_pandemia_TWh"]
    - df_comparacion["Gen_promedio_pandemia_TWh"]
)
df_comparacion["Cambio_porcentual"] = (
    (df_comparacion["Diferencia_TWh"] / df_comparacion["Gen_promedio_pandemia_TWh"]) * 100
)

# Ordenar por generacion post-pandemia (de mayor a menor)
df_comparacion = df_comparacion.sort_values(
    "Gen_promedio_post_pandemia_TWh", ascending=False
)

df_comparacion

## Paso 8: Detalle anio a anio (2020-2025)
Creamos una tabla pivote que muestra la generacion total de cada pais para cada anio del rango 2020-2025. Esto nos servira para ver la evolucion detallada.

In [ ]:
df_detalle = df_total_gen[
    df_total_gen["Year"].isin(PERIODO_PANDEMIA + PERIODO_POST_PANDEMIA)
][["Area", "ISO 3 code", "Year", "Generation (TWh)"]].copy()

pivot_detalle = df_detalle.pivot_table(
    index=["Area", "ISO 3 code"],
    columns="Year",
    values="Generation (TWh)"
).reset_index()

pivot_detalle

## Resumen de variables disponibles
Los siguientes DataFrames quedan en memoria para los proximos pasos (visualizacion con mapa):

| Variable | Descripcion |
|---|---|
| `df_americas` | Todos los datos de America (todas las fuentes, todos los anios) |
| `df_total_gen` | Solo generacion total por pais/anio |
| `df_pandemia` | Generacion total, periodo 2020-2022 |
| `df_post_pandemia` | Generacion total, periodo 2023-2025 |
| `df_comparacion` | Tabla comparativa con promedios y cambio porcentual |
| `pivot_detalle` | Tabla pivote anio a anio por pais |